In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator
from matplotlib import patheffects as pe, font_manager as fm
# from jamaica import plot
from matplotlib.ticker import MaxNLocator
import sys, pathlib
sys.path.append(str(pathlib.Path("../../../robyns_libraries").resolve()))
import Robyn_paper_2_defs


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
def compute_discount_rate (years, discount_rate): 
    discount_rates = [1/(1+discount_rate) ** y for y in range(years + 1)]
    discount_rates_sum = sum(discount_rates)
    return discount_rates_sum  
compute_discount_rate(50, 0.1)


In [ ]:
maintenance_discount = compute_discount_rate(3, 0.10) - 1
maintenance_discount

In [ ]:
catchments = gpd.read_file(base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg") #[["catchment_uid","geometry"]]

jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
cost_csv = (base_path / "dphil_paper_2/results/restoration_costs/catchment_priority_costs_max.csv")
costs = pd.read_csv(cost_csv)
costs.head()

In [ ]:
catchment_avoided_damages = pd.read_csv(base_path / "dphil_paper_2/results/flood_damage_results/expected_annual_damages_catchment/catchment_avoided_ead_max_usd.csv")
catchment_avoided_damages.head()

In [ ]:
# Ensure matching dtypes for the key
costs["catchment_uid"] = costs["catchment_uid"].astype(int)
catchment_avoided_damages["catchment_uid"] = catchment_avoided_damages["catchment_uid"].astype(int)

# If either table might have duplicate IDs, de-dup first (optional)
costs = costs.drop_duplicates(subset=["catchment_uid"])
catchment_avoided_damages = catchment_avoided_damages.drop_duplicates(subset=["catchment_uid"])

# Merge (one-to-one expected)
merged = costs.merge(
    catchment_avoided_damages,
    on="catchment_uid",
    how="left",
    validate="one_to_one"
)

# one-liner (also coerces the column to numeric just in case)
merged["avoided_ead_max_usd_discounted"] = (
    pd.to_numeric(merged["avoided_ead_max_usd"], errors="coerce") * compute_discount_rate(50, 0.10)
)

# one-liner (also coerces the column to numeric just in case)
merged["maintenance_usd_discounted"] = (
    pd.to_numeric(merged["one_year_maintenance_costs_USD"], errors="coerce") * (compute_discount_rate(3, 0.10) - 1)
)

print(merged.head())

# Quick check for any unmatched IDs
missing = merged["avoided_ead_max"].isna().sum()
print(f"Rows with no avoided_ead match: {missing}")

# Save for reuse
out_csv = base_path / "dphil_paper_2/results/bcr_mca_results/catchment_costs_and_avoided_ead_max.csv"
merged.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# after merged is defined and the column exists
total_avoided_max_usd_discounted = pd.to_numeric(
    merged["avoided_ead_max_usd_discounted"], errors="coerce"
).sum()

print(f"Total avoided EAD (discounted, max) = US$ {total_avoided_max_usd_discounted:,.2f}")
# If you also want millions:
print(f"…in millions: {total_avoided_max_usd_discounted / 1e6:,.2f} USD mn")


In [ ]:
# Add total = costs_USD + three_year_maintenance_costs_USD
merged["costs_plus_three_year_maintenance_USD"] = (
    pd.to_numeric(merged["costs_USD"], errors="coerce")
    + pd.to_numeric(merged["three_year_maintenance_costs_USD"], errors="coerce")
)

# (Optional) also in USD millions
merged["costs_plus_three_year_maintenance_USD_mn"] = (
    merged["costs_plus_three_year_maintenance_USD"] / 1e6
)

print(
    merged[
        ["catchment_uid", "costs_USD", "three_year_maintenance_costs_USD",
         "costs_plus_three_year_maintenance_USD",
         "costs_plus_three_year_maintenance_USD_mn"]
    ].head().to_string(index=False, formatters={
        "costs_USD": "{:,.2f}".format,
        "three_year_maintenance_costs_USD": "{:,.2f}".format,
        "costs_plus_three_year_maintenance_USD": "{:,.2f}".format,
        "costs_plus_three_year_maintenance_USD_mn": "{:,.2f}".format,
    })
)

# Rename the column
merged = merged.rename(columns={
    "costs_plus_three_year_maintenance_USD": "Total cost"
})


# Overwrite the CSV you already wrote
out_csv = base_path / "dphil_paper_2/results//bcr_mca_results/catchment_costs_and_avoided_ead_max_.csv"
merged.to_csv(out_csv, index=False)
print("Updated:", out_csv)

In [ ]:
# Ensure numeric (optional but robust)
merged["avoided_ead_max_usd"] = pd.to_numeric(merged["avoided_ead_max_usd"], errors="coerce")
merged["costs_USD"]           = pd.to_numeric(merged["costs_USD"], errors="coerce")

# Benefit–cost ratio = avoided_ead_max_usd / costs_USD
merged["benefit_cost_ratio"] = np.where(
    (merged["costs_USD"] > 0) & np.isfinite(merged["costs_USD"]),
    merged["avoided_ead_max_usd"] / merged["costs_USD"],
    np.nan
)

# Undiscounted BCR
merged["bcr_usd"] = merged["avoided_ead_max_usd"] / merged["costs_USD"]

# Discounted BCR: costs + discounted maintenance in the denominator
merged["total_cost_usd_discounted"] = merged["costs_USD"] + merged["maintenance_usd_discounted"]
merged["bcr_usd_discounted"] = merged["avoided_ead_max_usd_discounted"] / merged["total_cost_usd_discounted"]

# Replace divide-by-zero infinities with NaN (optional but handy)
merged.replace([np.inf, -np.inf], np.nan, inplace=True)

cols = ["catchment_uid", "avoided_ead_max_usd", "costs_USD", "bcr_usd", "bcr_usd_discounted"]
print(
    merged[cols]
      .sort_values("bcr_usd_discounted", ascending=False)
      .head(10)
      .to_string(index=False, formatters={
          "bcr_usd": lambda x: f"{x:,.2f}",
          "bcr_usd_discounted": lambda x: f"{x:,.2f}",
      })
)

# Save
out_csv = base_path / "dphil_paper_2/results//bcr_mca_results/catchment_costs_avoided_ead_with_bcr_max.csv"
merged.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
avg_bcr = pd.to_numeric(merged["bcr_usd"], errors="coerce").mean()
print(f"Average BCR (undiscounted): {avg_bcr:,.2f}")

avg_bcr_discounted = pd.to_numeric(merged["bcr_usd_discounted"], errors="coerce").mean()
print(f"Average BCR (discounted): {avg_bcr_discounted:,.2f}")


# Putting them on a map

In [ ]:
# === Map: Total cost by catchment (USD millions) ==============================

mm = globals().get("mm", 1/25.4)
out_dir = globals().get("out_dir", base_path / "figures")

# 1) Prepare data: take "Total cost" and convert to USD millions
if "Total cost" not in merged.columns:
    raise KeyError("Column 'Total cost' not found in 'merged'.")

tbl = merged[["catchment_uid", "Total cost"]].copy()
tbl["catchment_uid"] = pd.to_numeric(tbl["catchment_uid"], errors="coerce").astype("Int64")
tbl["total_cost_mn"] = pd.to_numeric(tbl["Total cost"], errors="coerce") / 1e6

# 2) Join to catchments geometry
gdf = catchments.merge(tbl[["catchment_uid", "total_cost_mn"]], on="catchment_uid", how="left")

# 3) Color scaling (linear in USD millions)
vals = gdf["total_cost_mn"].to_numpy(dtype="float64")
pos = vals[np.isfinite(vals) & (vals > 0)]
if pos.size:
    vmax = float(np.percentile(pos, 99.0))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0
else:
    vmax = 1.0
vmin = 0.0

# Sequential palette; darkest = highest cost
cmap = mpl.colormaps["magma_r"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))  # NaN light grey
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

# 4) Plot
with mpl.rc_context(globals().get("NATURE_RC", {})):
    fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
    ax.set_axis_off()

    # Fill + BLACK internal boundaries
    gdf.plot(
        ax=ax, column="total_cost_mn", cmap=cmap, norm=norm,
        edgecolor="black", linewidth=0.45, zorder=1
    )
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

    # Coastline casing (if you have jamaica_boundary)
    try:
        outline = getattr(jamaica_boundary, "union_all", getattr(jamaica_boundary, "unary_union"))()
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.2, zorder=98)
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
    except Exception:
        pass

    # Colorbar (USD millions)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)
    cbar.set_label("Total cost (US$ millions)", fontsize=6.5, labelpad=4)
    cbar.locator = MaxNLocator(nbins=6, integer=True)
    cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
    cbar.update_ticks()

    # Title
    ax.set_title("Total cost by catchment — establishment & three-year maintenance", fontsize=7, fontweight="bold", pad=6)

    # Labels (catchment IDs) with white halo
    NUM_FS = 5.0
    HALO_W = 0.75
    NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
    reps = gdf.geometry.representative_point()
    for (x, y, uid) in zip(reps.x, reps.y, gdf["catchment_uid"]):
        if pd.isna(uid):
            continue
        ax.text(
            x, y, str(int(uid)),
            fontproperties=NUM_FP, ha="center", va="center", color="black", zorder=20, snap=True,
            path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white", joinstyle="round", capstyle="round")]
        )

    # Save
    fname = base_path / "dphil_paper_2/results/Fig_total_cost_usd_millions_by_catchment"
    fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", fname.with_suffix(".png"))

In [ ]:
# === Map: avoided_ead_max_usd_discounted by catchment — Blues =================
# 1) Prepare data (USD millions)
if "avoided_ead_max_usd_discounted" not in merged.columns:
    raise KeyError("Column 'avoided_ead_max_usd_discounted' not found in 'merged'.")

tbl = merged[["catchment_uid", "avoided_ead_max_usd_discounted"]].copy()
tbl["catchment_uid"] = pd.to_numeric(tbl["catchment_uid"], errors="coerce").astype("Int64")
tbl["avoided_ead_mn"] = pd.to_numeric(tbl["avoided_ead_max_usd_discounted"], errors="coerce") / 1e6

# 2) Join to catchments geometry
gdf = catchments.merge(tbl[["catchment_uid", "avoided_ead_mn"]], on="catchment_uid", how="left")

# 3) Color scaling (linear, cap at 99th percentile)
vals = gdf["avoided_ead_mn"].to_numpy(dtype="float64")
pos = vals[np.isfinite(vals) & (vals > 0)]
if pos.size:
    vmax = float(np.percentile(pos, 99.0))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0
else:
    vmax = 1.0
vmin = 0.0

# --- Blues palette (dark = high). Use "Blues_r" if you want dark = low.
cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))  # NaN = light grey
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

# 4) Plot
with mpl.rc_context(globals().get("NATURE_RC", {})):
    fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
    ax.set_axis_off()

    # Polygons + BLACK boundaries
    gdf.plot(
        ax=ax, column="avoided_ead_mn", cmap=cmap, norm=norm,
        edgecolor="black", linewidth=0.45, zorder=1
    )
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

    # Coastline casing (if available)
    try:
        outline = getattr(jamaica_boundary, "union_all", getattr(jamaica_boundary, "unary_union"))()
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.2, zorder=98)
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
    except Exception:
        pass

    # Colorbar (USD millions)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)
    cbar.set_label("Avoided EAD (discounted, USD millions)", fontsize=6.5, labelpad=4)
    cbar.locator = MaxNLocator(nbins=6, integer=True)
    cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
    cbar.update_ticks()

    # Title
    ax.set_title("Avoided EAD (discounted) by catchment — USD millions (Blues)", fontsize=7, fontweight="bold", pad=6)

    # Labels (catchment IDs) with white halo
    NUM_FS = 5.0
    HALO_W = 0.75
    NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
    reps = gdf.geometry.representative_point()
    for (x, y, uid) in zip(reps.x, reps.y, gdf["catchment_uid"]):
        if pd.isna(uid): 
            continue
        ax.text(
            x, y, str(int(uid)),
            fontproperties=NUM_FP, ha="center", va="center", color="black", zorder=20, snap=True,
            path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white", joinstyle="round", capstyle="round")]
        )

    # Save
    fname = base_path / "dphil_paper_2/results/Fig_avoided_ead_max_usd_discounted_by_catchment_Blues"
    fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", fname.with_suffix(".png"))

In [ ]:
# Ensure key dtype matches (robust)
merged["catchment_uid"] = merged["catchment_uid"].astype(int)
catchments["catchment_uid"] = catchments["catchment_uid"].astype(int)

# Join and make a GeoDataFrame
joined_gdf = gpd.GeoDataFrame(
    merged.merge(catchments, on="catchment_uid", how="left"),
    geometry="geometry",
    crs=catchments.crs
)

# Quick sanity check
print("Rows with missing geometry:", joined_gdf["geometry"].isna().sum())

# Save for mapping/analysis
out_gpkg = base_path / "dphil_paper_2/results/catchment_costs_avoided_ead_with_bcr.gpkg"
joined_gdf.to_file(out_gpkg, layer="data", driver="GPKG")
print("Saved:", out_gpkg)

# (Optional) also save a CSV without geometry
joined_gdf.drop(columns="geometry").to_csv(base_path / "dphil_paper_2/results/catchment_costs_avoided_ead_with_bcr.csv", index=False)

In [ ]:
# === BCR map — crisper export variant (NEW CELL to compare) ==================
gdf = globals().get("joined_gdf", None)
if gdf is None:
    gdf = gpd.GeoDataFrame(
        merged.merge(catchments[["catchment_uid", "geometry"]], on="catchment_uid", how="left"),
        geometry="geometry",
        crs=catchments.crs
    )

# Fallback mm + out_dir
mm = globals().get("mm", 1/25.4)
out_dir = globals().get("out_dir", base_path / "figures")

# Values (unitless BCR) for LogNorm
vals = gdf["bcr_usd_discounted"].astype(float)
arr = vals.to_numpy()
pos = arr[np.isfinite(arr) & (arr > 0)]
FLOOR = 0.1
if pos.size:
    hi = np.percentile(pos, 98)
    lo = max(np.percentile(pos, 1), FLOOR)
else:
    hi, lo = 1.0, FLOOR
if hi <= lo:
    lo = hi/5 if hi > 0 else FLOOR
plot_vals = vals.where(vals > 0)

# Colormap + log scale
cmap = mpl.colormaps.get_cmap("Blues").copy()
cmap.set_bad("whitesmoke")
norm = mpl.colors.LogNorm(vmin=lo, vmax=hi)

# Preview DPI for notebook (sharper inline)
plt.rcParams["figure.dpi"] = 200

# Figure
fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
ax.set_axis_off()

# Base fill with neutral-grey internal boundaries
gdf.plot(ax=ax, column=plot_vals, cmap=cmap, norm=norm,
         edgecolor="#BDBDBD", linewidth=0.35, zorder=1)
# Reinforce boundaries on top (slightly darker & thicker)
gdf.boundary.plot(ax=ax, color="#9E9E9E", linewidth=0.50, zorder=16)

# Coastline casing (if available)
try:
    try:
        outline_geom = jamaica_boundary.union_all()
    except AttributeError:
        outline_geom = jamaica_boundary.unary_union
    gpd.GeoSeries([outline_geom], crs=gdf.crs).plot(
        ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98
    )
    gpd.GeoSeries([outline_geom], crs=gdf.crs).plot(
        ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99
    )
except Exception:
    pass

# Colorbar
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
nice = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50])
ticks = nice[(nice >= lo) & (nice <= hi)]
if ticks.size:
    cbar.set_ticks(ticks)
cbar.formatter = FuncFormatter(lambda x, pos: f"{x:g}")
cbar.update_ticks()
cbar.minorticks_off()
cbar.ax.yaxis.set_minor_locator(NullLocator())
cbar.ax.tick_params(which="both", width=0.35, length=2.0, labelsize=6)
cbar.set_label("Benefit–cost ratio", fontsize=6.5, labelpad=4)

# Title
ax.set_title(
    "Benefit–cost ratio by catchment (avoided EAD ÷ restoration costs)",
    fontsize=7, fontweight="bold", pad=6
)

# --- Scale bar + North arrow --------------------------------------------------
Robyn_paper_2_defs.draw_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)



# Labels (slightly larger for PNG readability)
NUM_FS, HALO_W = 6.0, 0.60
NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
pts = gdf.geometry.representative_point()
try:
    mask = gdf.to_crs(3448).geometry.area >= 2e6
except Exception:
    mask = ~pts.is_empty

texts = []
for (x, y, uid, show) in zip(pts.x, pts.y, gdf["catchment_uid"].astype(str), mask):
    if not show:
        continue
    t = ax.text(
        x, y, uid, fontproperties=NUM_FP, ha="center", va="center",
        color="black", zorder=20, snap=True,
        path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white",
                                    joinstyle="round", capstyle="round")]
    )
    texts.append(t)

# Declutter (if available)
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, only_move={'texts':'y'},
                expand_text=(1.08, 1.15), force_text=(0.06, 0.16))
except Exception:
    pass

plt.tight_layout()

# Save (higher-DPI PNG + vector PDF)
fname = base_path / "dphil_paper_2/results/Fig_BCR_by_catchment_grey_edges_crisp"
plt.savefig(fname.with_suffix(".png"), dpi=900, bbox_inches="tight", facecolor="white")
plt.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")
plt.show()
print("Saved:", fname.with_suffix(".png"))

# Optional: report PNG pixel size to embed at 100%
try:
    from PIL import Image
    w, h = Image.open(fname.with_suffix(".png")).size
    print("PNG pixels:", w, "x", h)
except Exception:
    pass

In [ ]:
# === Panel (vertical): Avoided EAD (disc.), Total cost, BCR — with labels =====
# Ensure join key types line up
catchments = catchments.copy()
catchments["catchment_uid"] = pd.to_numeric(catchments["catchment_uid"], errors="coerce").astype("Int64")
merged = merged.copy()
merged["catchment_uid"] = pd.to_numeric(merged["catchment_uid"], errors="coerce").astype("Int64")

# Metric CRS for consistent scaling
base = catchments.to_crs("EPSG:3448") if getattr(catchments.crs, "is_geographic", False) else catchments

# Common extent (+ small pad)
xmin, ymin, xmax, ymax = base.total_bounds
dx, dy = xmax - xmin, ymax - ymin
pad = 0.02
xlim = (xmin - pad*dx, xmax + pad*dx)
ylim = (ymin - pad*dy, ymax + pad*dy)

# Coastline outline (optional)
try:
    jb = jamaica_boundary.to_crs(base.crs) if getattr(jamaica_boundary, "crs", None) and base.crs else jamaica_boundary
    try:
        outline_geom = jb.union_all()
    except AttributeError:
        outline_geom = jb.unary_union
    outline_gs = gpd.GeoSeries([outline_geom], crs=base.crs)
except Exception:
    outline_gs = None

# --- Label positions: ONE point per catchment_uid (avoid duplicates) ----------
labels = (
    base[["catchment_uid", "geometry"]]
    .dropna(subset=["catchment_uid"])
    .dissolve(by="catchment_uid")  # merge multipart catchments
    .reset_index()
)
labels["pt"] = labels.geometry.representative_point()  # guaranteed inside polygon
labels["x"] = labels["pt"].x
labels["y"] = labels["pt"].y

# Optional: only label sufficiently large catchments (uncomment & adjust)
# areas_ha = labels.to_crs(3448).geometry.area / 10_000
# labels = labels[areas_ha >= 50]  # e.g., >= 50 ha

# --- Data per panel -----------------------------------------------------------
# (a) Avoided EAD discounted (USD mn, linear)
gdf_avoided = base.merge(merged[["catchment_uid", "avoided_ead_max_usd_discounted"]], on="catchment_uid", how="left")
gdf_avoided["avoided_mn"] = pd.to_numeric(gdf_avoided["avoided_ead_max_usd_discounted"], errors="coerce") / 1e6
av_vals = gdf_avoided["avoided_mn"].to_numpy(dtype="float64")
av_pos = av_vals[np.isfinite(av_vals) & (av_vals > 0)]
if av_pos.size:
    av_vmax = float(np.percentile(av_pos, 99.0))
    if not np.isfinite(av_vmax) or av_vmax <= 0:
        av_vmax = float(np.nanmax(av_pos)) if np.isfinite(np.nanmax(av_pos)) else 1.0
else:
    av_vmax = 1.0
av_vmin = 0.0

# (b) Total cost (USD mn, linear)
gdf_cost = base.merge(merged[["catchment_uid", "Total cost"]], on="catchment_uid", how="left")
gdf_cost["total_cost_mn"] = pd.to_numeric(gdf_cost["Total cost"], errors="coerce") / 1e6
tc_vals = gdf_cost["total_cost_mn"].to_numpy(dtype="float64")
tc_pos = tc_vals[np.isfinite(tc_vals) & (tc_vals > 0)]
if tc_pos.size:
    tc_vmax = float(np.percentile(tc_pos, 99.0))
    if not np.isfinite(tc_vmax) or tc_vmax <= 0:
        tc_vmax = float(np.nanmax(tc_pos)) if np.isfinite(np.nanmax(tc_pos)) else 1.0
else:
    tc_vmax = 1.0
tc_vmin = 0.0

# (c) BCR (log scale; positive only)
gdf_bcr = base.merge(merged[["catchment_uid", "bcr_usd_discounted"]], on="catchment_uid", how="left")
bcr_vals = gdf_bcr["bcr_usd_discounted"].astype(float).to_numpy()
bcr_pos = bcr_vals[np.isfinite(bcr_vals) & (bcr_vals > 0)]
FLOOR = 0.1
if bcr_pos.size:
    bcr_hi = float(np.percentile(bcr_pos, 98))
    bcr_lo = max(float(np.percentile(bcr_pos, 1)), FLOOR)
else:
    bcr_hi, bcr_lo = 1.0, FLOOR
if bcr_hi <= bcr_lo:
    bcr_lo = bcr_hi/5 if bcr_hi > 0 else FLOOR
gdf_bcr["bcr_plot"] = gdf_bcr["bcr_usd_discounted"].where(gdf_bcr["bcr_usd_discounted"] > 0)

# --- Colormaps/norms ----------------------------------------------------------
avoided_cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
avoided_norm = mpl.colors.Normalize(vmin=av_vmin, vmax=av_vmax)

cost_cmap = mpl.colormaps["magma_r"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
cost_norm = mpl.colors.Normalize(vmin=tc_vmin, vmax=tc_vmax)

bcr_cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
bcr_norm = mpl.colors.LogNorm(vmin=bcr_lo, vmax=bcr_hi)

# --- Figure (vertical stack) --------------------------------------------------
with mpl.rc_context(globals().get("NATURE_RC", {})):
    fig_w_mm = 180
    panel_h_mm = 60
    fig_h_mm = panel_h_mm * 3 + 8
    fig, axes = plt.subplots(3, 1, figsize=(fig_w_mm*mm, fig_h_mm*mm), constrained_layout=False)

    panels = [
        ("(a) Total avoided EAD (USD millions)", gdf_avoided, "avoided_mn",     avoided_cmap, avoided_norm, "USD millions"),
        ("(b) Total cost (USD millions)",              gdf_cost,    "total_cost_mn", cost_cmap,    cost_norm,    "USD millions"),
        ("(c) Benefit–cost ratio",        gdf_bcr,     "bcr_plot",      bcr_cmap,     bcr_norm,     "Benefit–cost ratio"),
    ]

    for ax, (title, gdf_plot, col, cmap, norm, cbar_label) in zip(axes, panels):
        ax.set_axis_off()
        ax.set_aspect("equal")

        # Polygons + BLACK boundaries
        gdf_plot.plot(ax=ax, column=col, cmap=cmap, norm=norm, edgecolor="black", linewidth=0.45, zorder=1)
        gdf_plot.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

        # Coastline casing
        if outline_gs is not None:
            try:
                outline_gs.to_crs(gdf_plot.crs).plot(ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98)
                outline_gs.to_crs(gdf_plot.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
            except Exception:
                pass

        # === UID labels (same positions for all panels) =======================
        for uid, x, y in zip(labels["catchment_uid"], labels["x"], labels["y"]):
            ax.text(
                x, y, f"{int(uid)}",
                ha="center", va="center",
                fontsize=5.0, color="#111111",
                zorder=120,
                path_effects=[pe.withStroke(linewidth=1.2, foreground="white")]
            )

        # Same extent for all
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # Title
        ax.set_title(title, fontsize=7, fontweight="bold", pad=4)

        # Colorbar
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.010)
        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.outline.set_linewidth(0.35)
        cbar.ax.tick_params(which="both", length=0, width=0, labelsize=6)
        cbar.set_label(cbar_label, fontsize=6.5, labelpad=4)

        if col == "bcr_plot":
            nice = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50])
            ticks = nice[(nice >= bcr_lo) & (nice <= bcr_hi)]
            if ticks.size:
                cbar.set_ticks(ticks)
            cbar.formatter = FuncFormatter(lambda x, pos: f"{x:g}")
            cbar.update_ticks()
        else:
            cbar.locator = MaxNLocator(nbins=6, integer=True)
            cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
            cbar.update_ticks()

    fig.subplots_adjust(left=0.02, right=0.98, top=0.97, bottom=0.04, hspace=0.12)

    # Save
    panel_path = base_path / "dphil_paper_2/results/panel_vertical_Avoided_Total_BCR_with_UID_labels"
    fig.savefig(panel_path.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", panel_path.with_suffix(".png"))

In [ ]:
# === Panel (vertical): Avoided EAD (disc.), Total cost, BCR — with labels =====
# Ensure join key types line up
catchments = catchments.copy()
catchments["catchment_uid"] = pd.to_numeric(catchments["catchment_uid"], errors="coerce").astype("Int64")
merged = merged.copy()
merged["catchment_uid"] = pd.to_numeric(merged["catchment_uid"], errors="coerce").astype("Int64")

# Metric CRS for consistent scaling
base = catchments.to_crs("EPSG:3448") if getattr(catchments.crs, "is_geographic", False) else catchments

# Common extent (+ small pad)
xmin, ymin, xmax, ymax = base.total_bounds
dx, dy = xmax - xmin, ymax - ymin
pad = 0.02
xlim = (xmin - pad*dx, xmax + pad*dx)
ylim = (ymin - pad*dy, ymax + pad*dy)

# Coastline outline (optional)
try:
    jb = jamaica_boundary.to_crs(base.crs) if getattr(jamaica_boundary, "crs", None) and base.crs else jamaica_boundary
    try:
        outline_geom = jb.union_all()
    except AttributeError:
        outline_geom = jb.unary_union
    outline_gs = gpd.GeoSeries([outline_geom], crs=base.crs)
except Exception:
    outline_gs = None

# --- Label positions: ONE point per catchment_uid (avoid duplicates) ----------
labels = (
    base[["catchment_uid", "geometry"]]
    .dropna(subset=["catchment_uid"])
    .dissolve(by="catchment_uid")  # merge multipart catchments
    .reset_index()
)
labels["pt"] = labels.geometry.representative_point()  # guaranteed inside polygon
labels["x"] = labels["pt"].x
labels["y"] = labels["pt"].y

# Optional: only label sufficiently large catchments (uncomment & adjust)
# areas_ha = labels.to_crs(3448).geometry.area / 10_000
# labels = labels[areas_ha >= 50]  # e.g., >= 50 ha

# --- Data per panel -----------------------------------------------------------
# (a) Avoided EAD discounted (USD mn, linear)
gdf_avoided = base.merge(merged[["catchment_uid", "avoided_ead_max_usd_discounted"]], on="catchment_uid", how="left")
gdf_avoided["avoided_mn"] = pd.to_numeric(gdf_avoided["avoided_ead_max_usd_discounted"], errors="coerce") / 1e6
av_vals = gdf_avoided["avoided_mn"].to_numpy(dtype="float64")
av_pos = av_vals[np.isfinite(av_vals) & (av_vals > 0)]
if av_pos.size:
    av_vmax = float(np.percentile(av_pos, 99.0))
    if not np.isfinite(av_vmax) or av_vmax <= 0:
        av_vmax = float(np.nanmax(av_pos)) if np.isfinite(np.nanmax(av_pos)) else 1.0
else:
    av_vmax = 1.0
av_vmin = 0.0

# (b) Total cost (USD mn, linear)
gdf_cost = base.merge(merged[["catchment_uid", "Total cost"]], on="catchment_uid", how="left")
gdf_cost["total_cost_mn"] = pd.to_numeric(gdf_cost["Total cost"], errors="coerce") / 1e6
tc_vals = gdf_cost["total_cost_mn"].to_numpy(dtype="float64")
tc_pos = tc_vals[np.isfinite(tc_vals) & (tc_vals > 0)]
if tc_pos.size:
    tc_vmax = float(np.percentile(tc_pos, 99.0))
    if not np.isfinite(tc_vmax) or tc_vmax <= 0:
        tc_vmax = float(np.nanmax(tc_pos)) if np.isfinite(np.nanmax(tc_pos)) else 1.0
else:
    tc_vmax = 1.0
tc_vmin = 0.0

# (c) BCR (log scale; positive only)
gdf_bcr = base.merge(merged[["catchment_uid", "bcr_usd_discounted"]], on="catchment_uid", how="left")
bcr_vals = gdf_bcr["bcr_usd_discounted"].astype(float).to_numpy()
bcr_pos = bcr_vals[np.isfinite(bcr_vals) & (bcr_vals > 0)]
FLOOR = 0.1
if bcr_pos.size:
    bcr_hi = float(np.percentile(bcr_pos, 98))
    bcr_lo = max(float(np.percentile(bcr_pos, 1)), FLOOR)
else:
    bcr_hi, bcr_lo = 1.0, FLOOR
if bcr_hi <= bcr_lo:
    bcr_lo = bcr_hi/5 if bcr_hi > 0 else FLOOR
gdf_bcr["bcr_plot"] = gdf_bcr["bcr_usd_discounted"].where(gdf_bcr["bcr_usd_discounted"] > 0)

# --- Colormaps/norms ----------------------------------------------------------
avoided_cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
avoided_norm = mpl.colors.Normalize(vmin=av_vmin, vmax=av_vmax)

cost_cmap = mpl.colormaps["magma_r"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
cost_norm = mpl.colors.Normalize(vmin=tc_vmin, vmax=tc_vmax)

bcr_cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
bcr_norm = mpl.colors.LogNorm(vmin=bcr_lo, vmax=bcr_hi)

# --- Figure (vertical stack) --------------------------------------------------
with mpl.rc_context(globals().get("NATURE_RC", {})):
    fig_w_mm = 180
    panel_h_mm = 60
    fig_h_mm = panel_h_mm * 3 + 8
    fig, axes = plt.subplots(3, 1, figsize=(fig_w_mm*mm, fig_h_mm*mm), constrained_layout=False)

    panels = [
        ("(a) Total avoided EAD (USD millions)", gdf_avoided, "avoided_mn",     avoided_cmap, avoided_norm, "USD millions"),
        ("(b) Total cost (USD millions)",              gdf_cost,    "total_cost_mn", cost_cmap,    cost_norm,    "USD millions"),
        ("(c) Benefit–cost ratio",        gdf_bcr,     "bcr_plot",      bcr_cmap,     bcr_norm,     "Benefit–cost ratio"),
    ]

    for ax, (title, gdf_plot, col, cmap, norm, cbar_label) in zip(axes, panels):
        ax.set_axis_off()
        ax.set_aspect("equal")

        # Polygons + BLACK boundaries
        gdf_plot.plot(ax=ax, column=col, cmap=cmap, norm=norm, edgecolor="black", linewidth=0.45, zorder=1)
        gdf_plot.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

        # Coastline casing
        if outline_gs is not None:
            try:
                outline_gs.to_crs(gdf_plot.crs).plot(ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98)
                outline_gs.to_crs(gdf_plot.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
            except Exception:
                pass

        # === UID labels (same positions for all panels) =======================
        for uid, x, y in zip(labels["catchment_uid"], labels["x"], labels["y"]):
            ax.text(
                x, y, f"{int(uid)}",
                ha="center", va="center",
                fontsize=5.0, color="#111111",
                zorder=120,
                path_effects=[pe.withStroke(linewidth=1.2, foreground="white")]
            )

        # Same extent for all
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # Title
        ax.set_title(title, fontsize=7, fontweight="bold", pad=4)

        # Scale bar + north arrow on the TOP panel only
        if ax is axes[0] and base.crs and base.crs.is_projected:
            pt = Robyn_paper_2_defs.add_scale_bar(
                ax, base, where="right-top",
                pad=0.07, length_km=20, max_frac=0.22,
                lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
            )
            if pt is not None:
                cx_data, cy_data = pt
                cx_ax, cy_ax = ax.transAxes.inverted().transform(
                    ax.transData.transform((cx_data, cy_data))
                )
                Robyn_paper_2_defs.add_north_arrow_axes(
                    ax, cx_ax, cy_ax,
                    size_frac=0.080, gap_frac=0.050,
                    shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
                    fs=5, lw=0.5
                )



        # Colorbar
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.010)
        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.outline.set_linewidth(0.35)
        cbar.ax.tick_params(which="both", length=0, width=0, labelsize=6)
        cbar.set_label(cbar_label, fontsize=6.5, labelpad=4)

        if col == "bcr_plot":
            nice = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50])
            ticks = nice[(nice >= bcr_lo) & (nice <= bcr_hi)]
            if ticks.size:
                cbar.set_ticks(ticks)
            cbar.formatter = FuncFormatter(lambda x, pos: f"{x:g}")
            cbar.update_ticks()
        else:
            cbar.locator = MaxNLocator(nbins=6, integer=True)
            cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
            cbar.update_ticks()

    fig.subplots_adjust(left=0.02, right=0.98, top=0.97, bottom=0.04, hspace=0.12)

    # Save
    panel_path = base_path / "dphil_paper_2/results/panel_vertical_Avoided_Total_BCR_with_UID_labels"
    fig.savefig(panel_path.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", panel_path.with_suffix(".png"))

In [ ]:

df = (
    merged[["catchment_uid", "bcr_usd_discounted"]]
    .dropna()
    .query("bcr_usd_discounted > 0")
    .sort_values("bcr_usd_discounted", ascending=False)
    .reset_index(drop=True)
)
df["rank"] = df.index + 1

N = 56  # top N to show
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df["rank"].iloc[:N], df["bcr_usd_discounted"].iloc[:N], color="steelblue", width=0.9)

# label bars with catchment_uid
for x, uid, val in zip(df["rank"].iloc[:N], df["catchment_uid"].iloc[:N], df["bcr_usd_discounted"].iloc[:N]):
    ax.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7, rotation=0)

ax.set_xlabel("Catchments ranked by BCR")
ax.set_ylabel("BCR (discounted)")
ax.set_title(f"Top {N} catchments by BCR")
ax.set_xlim(0.5, N + 0.5)
plt.tight_layout()
plt.show()


In [ ]:
forest_stats_path = base_path / "dphil_paper_2/results/catchment_attributes/catchment_landuse_by_category_stats.csv"
forest_stats = pd.read_csv(forest_stats_path)

In [ ]:
forest_stats.head()

In [ ]:

# forest_stats is your CSV as a DataFrame
# merged has bcr_usd_discounted from the notebook
df = (
    merged[["catchment_uid", "bcr_usd_discounted"]]
    .merge(
        forest_stats[["catchment_uid", "reforestable_of_catchment_pct"]],
        on="catchment_uid",
        how="left",
    )
    .dropna(subset=["bcr_usd_discounted"])
    .query("bcr_usd_discounted > 0")
    .sort_values("bcr_usd_discounted", ascending=False)
    .reset_index(drop=True)
)


In [ ]:

# Columns we need
bcr_col = "bcr_usd_discounted"
pct_col = "reforestable_of_catchment_pct"

# Merge BCRs (from `merged`) with reforestable % (from `forest_stats` CSV)
plot_df = (
    merged[["catchment_uid", bcr_col]]
    .merge(
        forest_stats[["catchment_uid", pct_col]],
        on="catchment_uid",
        how="left",
    )
    .dropna(subset=[bcr_col])
    .query(f"{bcr_col} > 0")
    .sort_values(bcr_col, ascending=False)
    .reset_index(drop=True)
)
plot_df["rank"] = plot_df.index + 1

# Ranked bar + secondary axis for % reforestable
N = 30  # top N to display
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(plot_df["rank"].iloc[:N], plot_df[bcr_col].iloc[:N], width=0.9, color="steelblue")
ax1.set_xlabel("Catchments ranked by BCR")
ax1.set_ylabel("BCR (discounted)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.set_xlim(0.5, N + 0.5)
ax1.set_title(f"Top {N} catchments by BCR with reforestable % overlay")

# Label bars with catchment_uid
for x, uid, val in zip(plot_df["rank"].iloc[:N], plot_df["catchment_uid"].iloc[:N], plot_df[bcr_col].iloc[:N]):
    ax1.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7, rotation=90)

# Overlay reforestable % on secondary axis
ax2 = ax1.twinx()
ax2.plot(plot_df["rank"].iloc[:N], plot_df[pct_col].iloc[:N], color="darkorange", marker="o")
ax2.set_ylabel("Reforestable area (% of catchment)", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")

fig.tight_layout()
plt.show()


In [ ]:

# Columns to use
bcr_col = "bcr_usd_discounted"           # from your merged table
pct_col = "reforestable_of_catchment_pct"  # from forest_stats CSV

# Merge BCRs (from `merged`) with % reforestable (from `forest_stats`)
plot_df = (
    merged[["catchment_uid", bcr_col]]
    .merge(
        forest_stats[["catchment_uid", pct_col]],
        on="catchment_uid",
        how="left",
    )
    .dropna(subset=[bcr_col, pct_col])
    .query(f"{bcr_col} > 0")
    .copy()
)

# Ranked by BCR
bcr_df = plot_df.sort_values(bcr_col, ascending=False).reset_index(drop=True)
bcr_df["rank"] = bcr_df.index + 1

# Ranked by % reforestable
pct_df = plot_df.sort_values(pct_col, ascending=False).reset_index(drop=True)
pct_df["rank"] = pct_df.index + 1

N = 30  # top N to show

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# BCR ranked histogram
ax = axes[0]
ax.bar(bcr_df["rank"].iloc[:N], bcr_df[bcr_col].iloc[:N], color="steelblue", width=0.9)
for x, uid, val in zip(bcr_df["rank"].iloc[:N], bcr_df["catchment_uid"].iloc[:N], bcr_df[bcr_col].iloc[:N]):
    ax.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7, rotation=0)
ax.set_xlabel("Catchments ranked by BCR")
ax.set_ylabel("BCR (discounted)")
ax.set_title(f"Top {N} by BCR")
ax.set_xlim(0.5, N + 0.5)

# % reforestable ranked histogram
ax = axes[1]
ax.bar(pct_df["rank"].iloc[:N], pct_df[pct_col].iloc[:N], color="darkorange", width=0.9)
for x, uid, val in zip(pct_df["rank"].iloc[:N], pct_df["catchment_uid"].iloc[:N], pct_df[pct_col].iloc[:N]):
    ax.text(x, val, str(int(uid)), ha="center", va="bottom", fontsize=7, rotation=0)
ax.set_xlabel("Catchments ranked by % reforestable")
ax.set_ylabel("% of catchment reforestable")
ax.set_title(f"Top {N} by reforestable %")
ax.set_xlim(0.5, N + 0.5)

fig.tight_layout()
plt.show()
